# Demonstration 
This example demonstrates the usage of the `lammpsparser` LAMMPS parser for simple interatomic potential calculations. Starting with the import of the required packages. In addition to the `lammpsparser` package and the atomic simulation environment `ase`, the `os` package and the `subprocess` package from the Python standard library are imported to create the working directory and execute the LAMMPS executable.

In [ ]:
import os
import subprocess

from ase.build import bulk
from lammpsparser import parse_lammps_output_files, write_lammps_structure

As a first step the working directory is created, in this example it is named `demo`: 

In [ ]:
working_directory = "demo"
os.makedirs(working_directory, exist_ok=True)

In addition to the atomistic structure, LAMMPS also requires an input script. These input scripts can vary in complexity, as they can include for loops and other logical structures. For simplicity, the example here is based on just a single `run` command, resulting in a static evaluation of the atomistic structure. For the example the [NiAlH_jea.eam.alloy](https://github.com/lammps/lammps/blob/develop/potentials/NiAlH_jea.eam.alloy) potential is used which is available as part of the LAMMPS repository. Furthermore, the `dump` commands and `thermo` commands for `lammpsparser` are already specified in this input script. 

In [ ]:
lammps_input_script = """\
units metal
dimension 3
boundary p p p
atom_style atomic

read_data lammps.data

pair_style eam/alloy
pair_coeff * * ../../binder/NiAlH_jea.eam.alloy Ni Al H

dump 1 all custom 100 dump.out id type xsu ysu zsu fx fy fz vx vy vz
dump_modify 1 sort id format line "%d %d %20.15g %20.15g %20.15g %20.15g %20.15g %20.15g %20.15g %20.15g %20.15g"
thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
thermo 100

run 0
"""

with open(os.path.join(working_directory, "lmp.in"), "w") as f:
    f.writelines(lammps_input_script)

After the input script is written the next step is to create the `lammps.data` file which contains the atomistic structure. Here a cubic bulk Aluminium structure is written to the input file. 

In [ ]:
structure = bulk("Al", cubic=True)
potential_elements = ["Ni", "Al", "H"]

write_lammps_structure(
    structure=structure,
    potential_elements=potential_elements,
    units="metal",
    file_name="lammps.data",
    working_directory=working_directory,
)

The structure parameter refers to the `ase.atoms.Atoms` structure and the `potential_elements` refers to the list of elements implemented in the specific interatomic potential. For example the `NiAlH_jea.eam.alloy` potential implements the elements Ni, Al and H, so when writing a structure for a simulation with this potential the `potential_elements=["Ni", "Al", "H"]`. It is important to maintain the order of the elements as LAMMPS internally references the elements based on their index, starting from one. The `units` parameter refers to the LAMMPS internal [units](https://docs.lammps.org/units.html) to convert the `ase.atoms.Atoms` object which is defined in Angstrom to the length scale of the LAMMPS simulation. Finally, file_name parameter and the current working directory `working_directory` parameter are designed to select the location where the LAMMPS structure should be written. With the default parameters the LAMMPS structure is written in the `lammps.data` file in the current directory.

Once all the input files are written to the `working_directory` then the LAMMPS calculation can be executed in an external Process using the `subprocess` package:

In [ ]:
command = "mpiexec -n 1 --oversubscribe lmp_mpi -in lmp.in"
output = subprocess.check_output(
    command,
    cwd=working_directory,
    shell=True,
    universal_newlines=True,
    env=os.environ.copy(),
)
output.split("\n")

In addition to the raw output returned by the LAMMPS simulation code, LAMMPS also writes multiple output files. These can be parsed with the `parse_lammps_output_files()` function:

In [ ]:
output = parse_lammps_output_files(
    working_directory=working_directory,
    structure=structure,
    potential_elements=potential_elements,
    units="metal",
    dump_h5_file_name="dump.h5",
    dump_out_file_name="dump.out",
    log_lammps_file_name="log.lammps",
)
output

In analogy to the `write_lammps_structure()` function the `working_directory` parameter refers to the directory which contains the output files. The `structure` parameter refers to the `ase.atoms.Atoms` object which should be used as template to parse the structure from the dump files. This structure is again required as LAMMPS internally references elements only by an index, so the template structure is required to map the elements from the interatomic potential back to the elements of the `ase.atoms.Atoms` object. In the same way the `potential_elements` refers to the list of elements implemented in the specific interatomic potential. The `units` parameter refers to the LAMMPS internal [units](https://docs.lammps.org/units.html) to convert the `ase.atoms.Atoms` object which is defined in Angstrom to the length scale of the LAMMPS simulation. Finally, the parameters `dump_h5_file_name`, `dump_out_file_name` and `log_lammps_file_name` refer to the output file names.

## Output

The dictionary returned by `parse_lammps_output_files()` has two top-level keys:

* **`"generic"`** — quantities converted to pyiron/ASE units (Å, eV, ps, …):
  * `steps` — simulation step indices
  * `natoms` — number of atoms per frame
  * `cells` — simulation cell tensors, shape `(n_frames, 3, 3)`
  * `positions` / `unwrapped_positions` — Cartesian atomic positions, shape `(n_frames, n_atoms, 3)`
  * `forces` — per-atom forces in eV/Å, shape `(n_frames, n_atoms, 3)`
  * `velocities` — per-atom velocities, shape `(n_frames, n_atoms, 3)`
  * `indices` — per-atom species indices into the structure's species list
  * `temperature` — instantaneous temperature in K
  * `energy_pot` — potential energy in eV
  * `energy_tot` — total energy in eV
  * `volume` — simulation cell volume in Å³
  * `pressures` — symmetric 3×3 pressure tensors in GPa, shape `(n_frames, 3, 3)`
* **`"lammps"`** — raw LAMMPS-specific thermo columns that have no generic equivalent.

All quantities from the `"generic"` key can be accessed directly:

In [ ]:
print("Potential energy (eV):", output["generic"]["energy_pot"])
print("Forces (eV/Ang):\n", output["generic"]["forces"])
print("Pressure tensor (GPa):\n", output["generic"]["pressures"])

The parser prefers an [H5MD](https://www.nongnu.org/h5md/) dump file (`dump.h5`) when present, and falls back to the plain-text dump (`dump.out`). H5MD is a portable HDF5-based format for molecular dynamics data. When only the text dump is available, the parser automatically handles the rotation of vectors from the LAMMPS upper-triangular cell frame back to the ASE coordinate frame using the `UnfoldingPrism` class. If neither file exists, a `FileNotFoundError` is raised.

## Structure

The `write_lammps_structure()` function (aliased from `write_lammps_datafile` in `lammpsparser.structure`) writes an ASE `Atoms` object to a LAMMPS data file. It supports several [atom_style](https://docs.lammps.org/atom_style.html) modes:

In [ ]:
from lammpsparser import write_lammps_structure

# Default: atom_style atomic
write_lammps_structure(
    structure=structure,
    potential_elements=["Ni", "Al", "H"],
    units="metal",
    file_name="lammps.data",
    working_directory=working_directory,
    atom_type="atomic",
)

The full parameter reference:

| Parameter | Default | Description |
|---|---|---|
| `structure` | — | ASE `Atoms` object to write |
| `potential_elements` | — | Ordered list of element symbols matching the potential definition |
| `units` | `"metal"` | LAMMPS [unit system](https://docs.lammps.org/units.html); controls length and mass conversion |
| `file_name` | `"lammps.data"` | Output filename |
| `working_directory` | `None` | Directory to write the file into |
| `atom_type` | `"atomic"` | LAMMPS [`atom_style`](https://docs.lammps.org/atom_style.html): `"atomic"`, `"charge"`, `"bond"`, or `"full"` |
| `bond_dict` | `None` | Bond topology dict, needed for `"bond"` and `"full"` atom styles |

Internally, the function constructs an `UnfoldingPrism` from the cell to convert from ASE's arbitrary cell representation to the upper-triangular cell format required by LAMMPS. For triclinic cells this involves a rotation of all positions and velocities. If the structure carries non-zero velocities they are written to the data file as well.

## Potential

The `lammpsparser.potential` sub-module provides helper functions to discover and validate interatomic potentials from the [NIST Interatomic Potentials Repository](https://www.ctcms.nist.gov/potentials/) (via the `iprpy-data` conda package). The three public functions exported from `lammpsparser` are `get_potential_dataframe`, `get_potential_by_name`, and `validate_potential_dataframe`.

### `get_potential_dataframe`

Returns all potentials compatible with the chemical species present in a given structure:

In [ ]:
from lammpsparser import get_potential_dataframe

al_structure = bulk("Al", cubic=True)
df_potentials = get_potential_dataframe(structure=al_structure)
df_potentials[["Name", "Species", "Model"]].head()

The function automatically locates the `iprpy` resource directory using the `CONDA_PREFIX` environment variable. An explicit path can be provided via the `resource_path` parameter. The returned `DataFrame` has its `"Config"` column populated with absolute paths to the potential files so the entries can be passed directly to LAMMPS without further path manipulation.

### `get_potential_by_name`

Retrieves a single potential row by its exact name string from the NIST/OpenKIM database:

In [ ]:
from lammpsparser import get_potential_by_name

potential = get_potential_by_name("2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1")
print(potential["Name"])
print(potential["Species"])
print(potential["Config"])

The returned `pandas.Series` contains a `"Config"` field — a list of LAMMPS input lines (typically `pair_style` and `pair_coeff` commands) that can be inserted into an input script directly.

### `validate_potential_dataframe`

When a potential selection may contain multiple rows, `validate_potential_dataframe` ensures exactly one potential is selected and returns it as a `pandas.Series`:

In [ ]:
from lammpsparser import get_potential_dataframe, validate_potential_dataframe

df = get_potential_dataframe(structure=al_structure)
# Filter to a single potential
df_single = df[df["Name"] == "2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1"]
potential_series = validate_potential_dataframe(df_single)
potential_series["Name"]

If the input is an empty `DataFrame` or contains more than one row, a `ValueError` is raised. This function is useful in workflows where the potential selection may be ambiguous and the user should be prompted to narrow it down.

## Compatibility Interface

The `lammpsparser.compatibility` sub-module provides a higher-level functional interface that mirrors the LAMMPS job API from the [pyiron_atomistics](https://github.com/pyiron/pyiron_atomistics) package. Where `pyiron_atomistics` wraps LAMMPS in an object-oriented job class:

```python
# pyiron_atomistics approach
job = pr.create.job.Lammps("lmp")
job.structure = structure
job.potential = "1995--Angelo-J-E--Ni-Al-H--LAMMPS--ipr1"
job.calc_md(temperature=500.0, pressure=0.0, n_ionic_steps=1000, seed=12345)
job.run()
job["output/generic/temperature"]
```

`lammpsparser` provides the same calculation logic as plain Python functions without the `pyiron_atomistics` dependency. The three calculation modes — static, MD, and minimisation — are available via `calc_static`, `calc_md`, and `calc_minimize` respectively, and are bundled together in the high-level `lammps_file_interface_function`.

### `lammps_file_initialization`

The `lammps_file_initialization` function generates the preamble of a LAMMPS input script — the `units`, `dimension`, `boundary`, `atom_style`, and `read_data` commands. Boundary conditions are derived automatically from the `pbc` attribute of the ASE `Atoms` object (`p` for periodic, `f` for fixed), matching the [boundary command](https://docs.lammps.org/boundary.html) documentation:

In [ ]:
from lammpsparser import lammps_file_initialization

init_lines = lammps_file_initialization(
    structure=bulk("Al", cubic=True),
    units="metal",
)
print("\n".join(init_lines))

When restarting a calculation from a binary [restart file](https://docs.lammps.org/restart.html) the `read_restart_file=True` flag replaces the `read_data` command with `read_restart`:

In [ ]:
restart_lines = lammps_file_initialization(
    structure=bulk("Al", cubic=True),
    units="metal",
    read_restart_file=True,
    restart_file="restart.out",
)
print("\n".join(restart_lines))

### `calc_static`, `calc_md`, `calc_minimize`

The three calculation functions generate the LAMMPS input lines for each calculation mode. They all return lists of strings to be written to the LAMMPS input script.

#### Static calculation

`calc_static` performs a single-point energy evaluation — `run 0` in LAMMPS terminology — which computes forces, stresses, and energies for the current atomic configuration without moving atoms:

In [ ]:
from lammpsparser import calc_static

print("\n".join(calc_static()))

#### Molecular dynamics

`calc_md` generates the thermostat and barostat commands for a molecular dynamics run. By default it uses the [Nosé-Hoover](https://docs.lammps.org/fix_nh.html) algorithm. The ensemble is selected by the combination of `temperature` and `pressure` parameters:

| `temperature` | `pressure` | Ensemble |
|---|---|---|
| `None` | `None` | NVE |
| float | `None` | NVT |
| float | float/list | NPT |

All time-like values (`time_step`, `temperature_damping_timescale`, `pressure_damping_timescale`) are given in femtoseconds and are automatically rescaled to the target LAMMPS [unit system](https://docs.lammps.org/units.html):

In [ ]:
from lammpsparser import calc_md

# NVT run at 500 K
nvt_lines = calc_md(
    temperature=500.0,
    n_print=100,
    time_step=1.0,
    seed=12345,
    units="metal",
)
print("\n".join(nvt_lines))

In [ ]:
# NPT run at 500 K, 0 GPa (isotropic)
npt_lines = calc_md(
    temperature=500.0,
    pressure=0.0,
    n_print=100,
    time_step=1.0,
    seed=12345,
    units="metal",
)
print("\n".join(npt_lines))

Pressure can also be specified as a list of up to 6 components `[pxx, pyy, pzz, pxy, pxz, pyz]` in GPa, with `None` for unconstrained directions. Setting `langevin=True` switches to [Langevin dynamics](https://docs.lammps.org/fix_langevin.html) for the thermostat.

#### Geometry optimisation

`calc_minimize` generates the input lines for a geometry optimisation using LAMMPS's [minimize command](https://docs.lammps.org/minimize.html). The default algorithm is conjugate gradient (`style="cg"`):

In [ ]:
from lammpsparser import calc_minimize

minimize_lines, _ = calc_minimize(
    structure=bulk("Al", cubic=True),
    ionic_force_tolerance=1e-4,
    max_iter=100000,
    n_print=100,
    units="metal",
)
print("\n".join(minimize_lines))

Setting `pressure` to a non-`None` value adds a `fix box/relax` command to allow the simulation cell to relax, enabling a full cell + geometry optimisation (the LAMMPS equivalent of `ISIF=3` in VASP).

### `lammps_file_interface_function`

The `lammps_file_interface_function` is the single entry point that orchestrates the complete LAMMPS workflow: it writes the input script, writes the structure data file, executes LAMMPS, and parses the output. Its interface is closely modelled on the LAMMPS job from `pyiron_atomistics` — the same parameter names, the same potential name strings from the NIST/OpenKIM database, and the same output dictionary structure.

The key difference is that `lammps_file_interface_function` is a plain function that returns a 3-tuple `(shell_output, parsed_output, job_crashed)` rather than a job object with persistence. This makes it straightforward to embed in scripts, notebooks, and workflow engines without installing `pyiron_atomistics`.

**Static calculation:**

In [ ]:
from lammpsparser import lammps_file_interface_function

shell_output, parsed_output, job_crashed = lammps_file_interface_function(
    working_directory=os.path.abspath("demo_static"),
    structure=bulk("Al", cubic=True),
    potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1",
    calc_mode="static",
)
print("Energy (eV):", parsed_output["generic"]["energy_pot"])

**Molecular dynamics calculation:**

In [ ]:
shell_output, parsed_output, job_crashed = lammps_file_interface_function(
    working_directory=os.path.abspath("demo_md"),
    structure=bulk("Al", cubic=True),
    potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1",
    calc_mode="md",
    calc_kwargs={
        "temperature": 500.0,
        "pressure": 0.0,
        "n_ionic_steps": 1000,
        "n_print": 100,
        "seed": 12345,
    },
)
print("Mean temperature (K):", parsed_output["generic"]["temperature"].mean())

**Geometry optimisation:**

In [ ]:
shell_output, parsed_output, job_crashed = lammps_file_interface_function(
    working_directory=os.path.abspath("demo_minimize"),
    structure=bulk("Al", cubic=True),
    potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1",
    calc_mode="minimize",
    calc_kwargs={
        "ionic_force_tolerance": 1e-4,
        "pressure": 0.0,
    },
)
print("Final energy (eV):", parsed_output["generic"]["energy_pot"][-1])

Instead of passing `calc_kwargs` as a plain dictionary, the typed dataclasses `CalcMDInput` and `CalcMinimizeInput` can be used. They provide IDE auto-completion and make the parameter set explicit:

In [ ]:
from lammpsparser.compatibility.data import CalcMDInput

md_input = CalcMDInput(
    temperature=500.0,
    pressure=0.0,
    n_ionic_steps=1000,
    n_print=100,
    seed=12345,
)

shell_output, parsed_output, job_crashed = lammps_file_interface_function(
    working_directory=os.path.abspath("demo_md_dataclass"),
    structure=bulk("Al", cubic=True),
    potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1",
    calc_dataclass=md_input,
)
print("Mean temperature (K):", parsed_output["generic"]["temperature"].mean())

The full parameter reference for `lammps_file_interface_function`:

| Parameter | Default | Description |
|---|---|---|
| `working_directory` | — | Directory where input and output files are written |
| `structure` | — | ASE `Atoms` object |
| `potential` | — | Potential name string, `pandas.Series`, or `pandas.DataFrame` |
| `calc_mode` | `"static"` | Calculation mode: `"static"`, `"md"`, or `"minimize"` |
| `calc_kwargs` | `None` | Dict of keyword arguments for the selected `calc_mode` |
| `calc_dataclass` | `None` | `CalcMDInput` or `CalcMinimizeInput` dataclass (overrides `calc_mode` and `calc_kwargs`) |
| `units` | `"metal"` | LAMMPS [unit system](https://docs.lammps.org/units.html) |
| `lmp_command` | `None` | LAMMPS executable command; defaults to `ASE_LAMMPSRUN_COMMAND` env variable or `mpiexec -n 1 --oversubscribe lmp_mpi -in lmp.in` |
| `resource_path` | `None` | Path to the `iprpy` resource directory for potential lookup |
| `input_control_file` | `None` | Dict of LAMMPS command overrides applied to the generated input script |
| `write_restart_file` | `False` | Append a `write_restart` command at the end of the run |
| `read_restart_file` | `False` | Start from a binary [restart file](https://docs.lammps.org/restart.html) instead of `lammps.data` |
| `restart_file` | `"restart.out"` | Filename of the LAMMPS restart file |

The `input_control_file` parameter accepts a dict mapping LAMMPS command keywords to replacement value strings. Any key matching an existing command in the generated script replaces that line; unmatched keys are appended at the end. This provides a lightweight way to override individual input parameters without rewriting the full script generation logic.

### Comparison with the pyiron_atomistics LAMMPS Job

The table below summarises the correspondence between the `pyiron_atomistics` LAMMPS job and `lammps_file_interface_function`:

| Aspect | `pyiron_atomistics` LAMMPS job | `lammps_file_interface_function` |
|---|---|---|
| Interface style | Object-oriented (`job.calc_md(...)`, `job.run()`) | Functional (single call, returns 3-tuple) |
| Structure input | `job.structure = structure` | `structure=` keyword argument |
| Potential | `job.potential = "name"` | `potential="name"` keyword argument |
| Calculation mode | `job.calc_static()` / `job.calc_md(...)` / `job.calc_minimize(...)` | `calc_mode="static"/"md"/"minimize"` + `calc_kwargs` |
| Output access | `job["output/generic/temperature"]` | `parsed_output["generic"]["temperature"]` |
| Output keys | Same `"generic"` / `"lammps"` structure | Identical |
| Potential names | NIST/OpenKIM strings | Same NIST/OpenKIM strings |
| Unit handling | Automatic | Automatic via `units=` parameter |
| Persistence | HDF5 via `pyiron_base` | None (plain dict) |
| Dependencies | `pyiron_atomistics`, `pyiron_base` | `lammpsparser`, ASE |

Because the output dictionary structure is identical, code that processes `job["output/generic/..."]` values from a `pyiron_atomistics` job can typically consume `parsed_output["generic"]["..."]` from `lammps_file_interface_function` with minimal changes — only the access syntax differs.